<a href="https://colab.research.google.com/github/berinbalci/BGT208-Yapay-Zeka-Siber-Guvenlik/blob/main/Hafta-05/BGT208_Hafta05_Siniflandirma_Modelleri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BGT208 – Yapay Zekâ ve Siber Güvenlik
## Hafta 5 – Sınıflandırma Modelleri

**Amaç:** Aynı oltalama (phishing) veri seti üzerinde iki farklı sınıflandırma modeli oluşturmak, eğitmek ve tahminlerini görmek.

Bu hafta adım adım ilerleyeceğiz:
1. Kütüphaneleri tanıma
2. Veriyi Google Drive'dan yükleme
3. Özellikleri (X) ve hedefi (y) belirleme
4. Eğitim/test ayrımı
5. Logistic Regression modeli
6. Decision Tree modeli
7. Tahminleri yan yana inceleme

**Sınıflar:** `0 = Phishing (oltalama)`, `1 = Legitimate (güvenilir)`.

> Bu hafta performans metrikleri hesaplamıyoruz. Accuracy, Precision, Recall, F1 ve Confusion Matrix 6. haftanın konusu.

## 1. Adım – Kütüphaneleri içe aktarma

**Pandas:** CSV dosyasını tablo olarak okumamızı sağlar.

**train_test_split:** Veriyi eğitim ve test olarak ayırır.

**LogisticRegression:** İki sınıftan birini tahmin etmek için kullanılan sınıflandırma algoritmasıdır. Adında *regression* geçse de burada sınıflandırma yapar.

**DecisionTreeClassifier:** Sorulara ve koşullara dayalı kararlar oluşturan sınıflandırma algoritmasıdır.

**Pipeline, SimpleImputer, StandardScaler:** Eksik sayısal değerleri doldurmak ve gerektiğinde özellikleri benzer ölçeğe getirmek için kullanılır.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

## 2. Adım – Google Drive bağlantısı

Önceki haftalarda kullandığımız CSV dosyası Google Drive'daki `BGT208` klasöründe. Bu komut Drive'a erişim izni ister.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Adım – Veri setini yükleme

`pd.read_csv()` CSV dosyasını okur. `head()` ilk beş satırı gösterir; `shape` satır ve sütun sayısını verir.

In [3]:
dosya_yolu = "/content/drive/MyDrive/bg208-yz-bilgi/PhiUSIIL_Phishing_URL_Dataset 3.csv"
df = pd.read_csv(dosya_yolu)
print("Veri boyutu (satır, sütun):", df.shape)
df.head()

Veri boyutu (satır, sütun): (235795, 56)


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


## 4. Adım – X ve y'yi hazırlama

- **X (özellikler):** Modelin tahmin için kullanacağı sayısal bilgiler.
- **y (hedef):** Modelin tahmin etmeye çalıştığı `label` sütunu.

`URL` gibi metin sütunlarını bu temel uygulamada kullanmıyoruz. Metinleri sayıya dönüştürme (encoding) ayrı bir hazırlık gerektirir.

`FILENAME` varsa kayıt tanımlayıcısı olduğu için çıkarıyoruz. `label` sütununu X'e koymuyoruz; aksi hâlde model cevabı önceden görmüş olur.

**Not:** Buradaki otomatik sayısal sütun seçimi öğretim amaçlıdır; gerçek projede özelliklerin anlamı ve veri sızıntısı riski ayrıca incelenmelidir.

In [4]:
X = df.select_dtypes(include="number").drop(columns=["label", "FILENAME"], errors="ignore")
y = df["label"]

print("Özellik sayısı:", X.shape[1])
print("X boyutu:", X.shape)
print("y boyutu:", y.shape)
print("Sınıflar:")
print(y.value_counts())
X.head()

Özellik sayısı: 50
X boyutu: (235795, 50)
y boyutu: (235795,)
Sınıflar:
label
1    134850
0    100945
Name: count, dtype: int64


,URLLength,DomainLength,IsDomainIP,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,...,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef
0,31,24,0,100.0,1.000000,0.522907,0.061933,3,1,0,...,1,0,0,1,34,20,28,119,0,124
1,23,16,0,100.0,0.666667,0.032650,0.050207,2,1,0,...,0,0,0,1,50,9,8,39,0,217
2,29,22,0,100.0,0.866667,0.028555,0.064129,2,2,0,...,0,0,0,1,10,2,7,42,2,5
3,26,19,0,100.0,1.000000,0.522907,0.057606,3,1,0,...,0,1,1,1,3,27,15,22,1,31
4,33,26,0,100.0,1.000000,0.079963,0.059441,3,1,0,...,1,1,0,1,244,15,34,72,1,85


## 5. Adım – Eğitim ve test verisini ayırma

- **Eğitim (%80):** Modelin örüntüleri öğrenmesi için.
- **Test (%20):** Modelin eğitimde görmediği örneklerde tahmin yapması için.

`random_state=42` aynı ayrımın tekrar elde edilmesini sağlar. `stratify=y` iki sınıfın oranlarını eğitim ve testte yaklaşık korur.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Eğitim:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Eğitim: (188636, 50) (188636,)
Test: (47159, 50) (47159,)


## 6. Adım – Logistic Regression modelini oluşturma

Bu model özellikleri birlikte değerlendirerek örneğin `0` mı yoksa `1` mi olduğunu tahmin etmeye çalışır.

**Ön hazırlık neden var?** Bazı sayısal sütunlar eksik değer içerebilir veya farklı büyüklüklerde olabilir. `SimpleImputer` eksikleri **yalnızca eğitim verisinden öğrendiği medyanlarla** doldurur; `StandardScaler` sayısal değerleri ölçeklendirir. `Pipeline` bu işlemleri modelle birlikte sıralar ve test verisinden bilgi sızmasını önler.

`fit()` eğitir, `predict()` tahmin üretir.

In [6]:
lojistik_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)
lojistik_model.fit(X_train, y_train)
lojistik_tahmin = lojistik_model.predict(X_test)
print("Logistic Regression ilk 10 tahmin:", lojistik_tahmin[:10])

Logistic Regression ilk 10 tahmin: [1 1 1 1 1 0 1 1 1 0]


## 7. Adım – Decision Tree (Karar Ağacı) modelini oluşturma

Karar Ağacı, özellikler için koşullar oluşturarak sınıfa karar verir. Örneğin bir özelliğin belirli bir değerden büyük olup olmadığını kontrol eden dallanmalar oluşturabilir.

`max_depth=5` ağacın derinliğini sınırlar. Karar ağacında bu örnek için ölçeklendirme zorunlu değildir; eksik değerleri yine eğitim verisinden öğrenilen medyanlarla dolduruyoruz.

In [7]:
agac_modeli = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=5, random_state=42)
)
agac_modeli.fit(X_train, y_train)
agac_tahmin = agac_modeli.predict(X_test)
print("Decision Tree ilk 10 tahmin:", agac_tahmin[:10])

Decision Tree ilk 10 tahmin: [1 1 1 1 1 0 1 1 1 0]


## KNN (K-En Yakın Komşu)

KNN, yeni bir veriyi sınıflandırırken ona en çok benzeyen örneklere bakar.

Örneğin yeni bir web sitesinin en yakın 5 komşusundan 4'ü phishing ise model bu siteyi phishing olarak sınıflandırabilir.

- `n_neighbors=5`: En yakın 5 komşu incelenir.
- `StandardScaler()`: Özelliklerin ölçeklerini düzenler.
- `fit()`: Eğitim verilerini modele aktarır.
- `predict()`: Test verileri için sınıf tahmini yapar.


In [9]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

knn_model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)

knn_model.fit(X_train, y_train)

knn_pred = knn_model.predict(X_test)

print("KNN ilk 10 tahmin:")
print(knn_pred[:10])

KNN ilk 10 tahmin:
[1 1 1 1 1 0 1 1 1 0]


## Random Forest (Rastgele Orman)

Random Forest, birden fazla karar ağacının birlikte çalışmasıyla sınıflandırma yapar.

Tek bir karar ağacı yerine çok sayıda karar ağacının tahminlerinden yararlanır.

- `n_estimators=100`: Model 100 karar ağacı kullanır.
- `random_state=42`: Sonuçların tekrarlanabilir olmasını sağlar.
- `fit()`: Modeli eğitir.
- `predict()`: Test verileri için tahmin yapar.

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest ilk 10 tahmin:")
print(rf_pred[:10])

Random Forest ilk 10 tahmin:
[1 1 1 1 1 0 1 1 1 0]


## 8. Adım – Gerçek sınıflar ve tahminler

Şimdi ilk 10 test kaydı için **gerçek etiket** ile iki modelin **tahminini** yan yana gösterelim.

- `Gerçek`: Veri setindeki doğru sınıf.
- `Logistic Regression`: İlk modelin tahmini.
- `Decision Tree`: İkinci modelin tahmini.

Tabloda aynı veya farklı tahminler görebiliriz. **İlk 10 satıra bakarak hangi modelin daha iyi olduğuna karar vermiyoruz.** Bunun için 6. haftada performans ölçütleri kullanacağız.

In [13]:
sonuclar = pd.DataFrame({
    "Gerçek": y_test.iloc[:10].to_numpy(),
    "Logistic Regression": lojistik_tahmin[:10],
    "Decision Tree": agac_tahmin[:10],
    "KNN": knn_pred[:10],
    "Random Forest": rf_pred[:10]
})

sonuclar

,Gerçek,Logistic Regression,Decision Tree,KNN,Random Forest
0,1,1,1,1,1
1,1,1,1,1,1
2,1,1,1,1,1
3,1,1,1,1,1
4,1,1,1,1,1
5,0,0,0,0,0
6,1,1,1,1,1
7,1,1,1,1,1
8,1,1,1,1,1
9,0,0,0,0,0


## 9. Adım – Haftanın özeti

Bu hafta:
1. Aynı veri setini kullandık.
2. X ve y'yi belirledik.
3. Eğitim/test ayrımı yaptık.
4. **Logistic Regression** ve **Decision Tree** modellerini eğittik.
5. `predict()` ile tahminler oluşturduk.
6. Gerçek etiketlerle tahminleri yan yana inceledik.

**Temel akış:** Veri → X/y → Eğitim/Test → Model → `fit()` → `predict()`

### Öğrenciye düşünme soruları
- Modeli neden test verisiyle eğitmedik?
- `label` sütununu neden X'ten çıkardık?
- İki model aynı örnek için farklı tahmin verebilir mi?
- Yalnızca 10 tahmine bakmak neden yeterli değildir?

**Gelecek hafta:** Model Performansı ve Bias–Variance.